# MàthemaSharp.LinearAlgebra — Usage

LU with partial pivoting and QR for small real systems. Prerequisites: Release build, JupyterSharp on .NET 11. The LinearAlgebra library is net10.0, but the tests and this notebook also use the .NET 11 formats.

## Contracts
- LU: square, non-empty, finite matrix; rejects numerically small pivots. The factorisation is reusable and independent of the original matrix. Solve requires a finite, compatible right-hand side.
- Determinant can exceed the type's range; a small result is not enough to diagnose conditioning. Inverse solves for the columns of the identity; for Ax=b prefer Solve.
- LeastSquares requires rows ≥ columns, finite data and independent columns; it normalises the system, uses Householder and does not form XᵀX. Unrepresentable intermediates are reported.
- NormL2 and FrobeniusNorm use Hypot. Multiply/Identity accept INumber; LU, norms and QR require IEEE.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System.Numerics;
using System.Globalization;
using System.Runtime.InteropServices;
using MathemaSharp.LinearAlgebra;
Console.WriteLine(RuntimeInformation.FrameworkDescription);


.NET 11.0.0-rc.3.26522.148

## LU: pivoting, residual and reuse
The first pivot is zero: without a row swap the procedure cannot start. The expected solution is [1,2,3]. The residual is checked on the original data, not only on the expected solution.


In [ ]:
double[,] a = {{0,2,1},{1,1,0},{2,0,1}};
double[] b = [7,3,5];
var lu = LuFactorization<double>.Create(a);
var x = lu.Solve(b);
var reconstructed = MatrixAlgebra.Multiply(a,x);
var residual = reconstructed.Zip(b,(left,right)=>left-right).ToArray();
if (MatrixAlgebra.NormL2<double>(residual) > 1e-12) throw new Exception("LU residual");
if (Math.Abs(lu.Determinant()+4)>1e-12) throw new Exception("Determinant");
var identity = MatrixAlgebra.Multiply(a,lu.Inverse());
for(var i=0;i<3;i++) for(var j=0;j<3;j++) if(Math.Abs(identity[i,j]-(i==j?1:0))>1e-12) throw new Exception("Inverse");
var again = lu.Solve(b);
if (!again.SequenceEqual(x)) throw new Exception("Reuse");
Console.WriteLine("lu_solution: "+string.Join(", ",x));
Console.WriteLine("lu_residual: OK");
Console.WriteLine("inverse_and_reuse: OK");


lu_solution: 1, 2, 3
lu_residual: OK
inverse_and_reuse: OK

## QR in Decimal128, without conversions from double
The data are created from integers. The tolerances are expressed in T. A small exact case does not certify all matrices or all formats.


In [ ]:
var one = Decimal128.One;
var zero = Decimal128.Zero;
var two = Decimal128.CreateChecked(2);
var three = Decimal128.CreateChecked(3);
var five = Decimal128.CreateChecked(5);
Decimal128[,] design = {{one,zero},{zero,one},{one,one}};
var beta = MatrixAlgebra.LeastSquares<Decimal128>(design,[two,three,five]);
var eps = Decimal128.BitIncrement(one)-one;
if(Decimal128.Abs(beta[0]-two)>Decimal128.CreateChecked(64)*eps || Decimal128.Abs(beta[1]-three)>Decimal128.CreateChecked(64)*eps) throw new Exception("QR Decimal128");
Console.WriteLine("qr_decimal128: OK");
Console.WriteLine("norm_l2: "+MatrixAlgebra.NormL2<double>([3,4]).ToString(CultureInfo.InvariantCulture));


qr_decimal128: OK
norm_l2: 5

## Reference and limits
[Public API catalogue](12-api-reference.ipynb) · [Checks on the six formats](11-consolidation.ipynb) · [Inventory](../../ARCHITECTURE.md).

This library is not BLAS, does not automatically estimate conditioning and does not yet offer a general SVD. The rank/pivot thresholds can reject ill-conditioned matrices. Normalisation can fail if y/scale is not representable. For measurements use BenchmarkDotNet, not the notebook process.
